# Multi-Modal Genomic Transformer Model

This notebook implements a Transformer-based model designed for multi-task learning on genomic data. The model simultaneously predicts masked DNA nucleotides (a self-supervised task similar to Masked Language Modeling) and masked RNA expression states (a regression/classification task). This approach aims to leverage the interdependencies between DNA sequence and gene expression.

## 1. Imports and Global Constants

This section imports all necessary Python libraries and defines global constants used throughout the model. These constants include hyperparameters for the Transformer architecture, training parameters, and crucial values for the custom masking strategies.

In [1]:
import torch
import torch.nn as nn
import math
import copy
import random
import os
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, random_split, Dataset, Subset
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt

# --- NEW GLOBAL CONSTANTS FOR MASKING ---
# Original number of nucleotide types (A, C, G, T, N)
NUM_NUCLEOTIDES_ORIGINAL = 5

# Index used for the MASK token in the one-hot encoded DNA input.
# If original nucleotides are 0-4, MASK will be 5.
MASK_NUCLEOTIDE_INDEX = NUM_NUCLEOTIDES_ORIGINAL

# One-hot dimension including original nucleotides + MASK token
ONE_HOT_NUCLEOTIDE_DIM = NUM_NUCLEOTIDES_ORIGINAL + 1 # Total one-hot dimension (A,C,G,T,N,MASK = 6)

# Special value for masked RNA expression in the input (distinct from 0.0 or 1.0)
MASK_EXPRESSION_VALUE = 0.5

# Masking probabilities
DNA_MASK_PROBABILITY = 0.15
RNA_MASK_FRACTION = 0.50

# BERT-like masking strategy probabilities for DNA
BERT_MASK_REPLACEMENT_PROB = 0.8  # 80% of the time: replace with MASK_TOKEN
BERT_RANDOM_REPLACEMENT_PROB = 0.1 # 10% of the time: replace with random original nucleotide
BERT_ORIGINAL_REPLACEMENT_PROB = 0.1 # 10% of the time: keep original nucleotide

# Split ratios for chromosomes
TRAIN_CHROM_RATIO = 0.8
VAL_CHROM_RATIO = 0.1
TEST_CHROM_RATIO = 0.1 # Implicit, as it takes the rest

# Model Hyperparameters
# Total input features per position: ONE_HOT_NUCLEOTIDE_DIM (6) + 1 (for Expression) = 7
NEW_INPUT_FEATURES = ONE_HOT_NUCLEOTIDE_DIM + 1

D_MODEL = 128       # Dimension of the model's embeddings
NUM_HEADS = 4       # Number of attention heads
NUM_LAYERS = 3      # Number of Transformer encoder layers
D_FF = 256          # Dimension of the feed-forward network
DROPOUT = 0.1       # Dropout rate

# Dataset Parameters
DATA_SUBSET_RATIO = None # Set to a float (e.g., 0.1) for a subset; Set to None for full dataset
DATA_SUBSET_SIZE = 2000 # Set to an integer (e.g., 200000) for a subset; Set to None for full dataset
USE_CHROMOSOME_SPLIT = True # Use chromosome-based split if True, else random split

if DATA_SUBSET_RATIO is not None and (DATA_SUBSET_RATIO <= 0 or DATA_SUBSET_RATIO >= 1):
    raise ValueError("DATA_SUBSET_RATIO must be a float between 0 and 1 (exclusive). Use None for full dataset.")
if DATA_SUBSET_SIZE is not None and (DATA_SUBSET_SIZE <= 0 or not isinstance(DATA_SUBSET_SIZE, int)):
    raise ValueError("DATA_SUBSET_SIZE must be a positive integer. Use None for full dataset.")
if DATA_SUBSET_RATIO is not None and DATA_SUBSET_SIZE is not None:
    raise ValueError("Please set either DATA_SUBSET_RATIO or DATA_SUBSET_SIZE, not both. Use None for full dataset.")

# Training Hyperparameters
BATCH_SIZE = 16
LEARNING_RATE = 0.0001
NUM_EPOCHS = 50
GRADIENT_ACCUMULATION_STEPS = 4
WARMUP_STEPS = 100 # For learning rate scheduler

NUM_WORKERS = 2 # Number of data loading workers

# Placeholder for dynamically detected sequence length (window_size from data)
SEQ_LENGTH = None

ValueError: DATA_SUBSET_RATIO must be a float between 0 and 1 (exclusive). Use None for full dataset.

## 2. Utility Functions

This section includes helper functions that are not part of the core model but are essential for the overall pipeline, such as determining the correct data directory based on the execution environment.

In [4]:
# --- UTILITY FUNCTIONS ---

def get_data_dir():
    """
    Determines the correct data directory based on the execution environment.
    This function makes the code portable between Colab, local PyCharm, and cluster.
    """
    # Default to current working directory for local/cluster
    data_directory = os.path.join(os.getcwd(), 'data/')

    # Attempt to detect Google Colab and use Google Drive path
    try:
        from google.colab import drive
        drive.mount('/content/gdrive')
        # This path assumes your 'DnARnAProject' folder is directly in 'My Drive'
        google_drive_project_path = '/content/gdrive/MyDrive/DnARnAProject/'
        data_directory = os.path.join(google_drive_project_path, 'data/')
        print("Detected Google Colab environment. Using Google Drive path.")
    except ImportError:
        print("Not in Google Colab. Using local/cluster path.")

    if not os.path.isdir(data_directory):
        print(f"Error: The data directory '{data_directory}' does not exist.")
        print("Please ensure your data is located correctly (e.g., in a 'data/' folder relative to your script, or in Google Drive).")
        exit() # Exit if data directory is not found
    return data_directory

## 3. Transformer Core Modules

This section defines the fundamental building blocks of the Transformer architecture:
- **`MultiHeadAttention`**: Implements the core attention mechanism, allowing the model to weigh the importance of different parts of the input sequence.
- **`PositionWiseFeedForward`**: A simple two-layer feed-forward network applied independently to each position.
- **`PositionalEncoding`**: Adds information about the position of tokens in the sequence, as Transformers are permutation-invariant.

In [5]:
# --- MODEL MODULES ---

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            # Fill masked positions with a very small number for softmax
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        attn_probs = torch.softmax(attn_scores, dim=-1)
        output = torch.matmul(attn_probs, V)
        return output, attn_probs

    def split_heads(self, x):
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))

        attn_output, attn_probs = self.scaled_dot_product_attention(Q, K, V, mask)

        output = self.W_o(self.combine_heads(attn_output))
        return output, attn_probs


class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

## 4. Transformer Encoder Layer and Main Model

This section defines the full encoder layer of the Transformer and the main `DNASequenceClassifier` model. The `DNASequenceClassifier` is equipped with two distinct prediction heads for multi-task learning:
- One head for reconstructing masked DNA nucleotides.
- Another head for predicting masked RNA expression states.

In [6]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        attn_output, attn_probs = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output)) # Add & Norm for attention
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output)) # Add & Norm for feed-forward
        return x, attn_probs


class DNASequenceClassifier(nn.Module):
    def __init__(self, input_features, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout, num_nucleotides_original):
        """
        Initializes the DNASequenceClassifier for a masked language modeling (DNA)
        and masked expression prediction (RNA) task.

        Args:
            input_features (int): Number of input features per position.
                                  (ONE_HOT_NUCLEOTIDE_DIM + 1 for expression state)
            d_model (int): Dimension of the model's embeddings.
            num_heads (int): Number of attention heads.
            num_layers (int): Number of encoder layers.
            d_ff (int): Dimension of the feed-forward network.
            max_seq_length (int): Maximum sequence length for positional encoding.
            dropout (float): Dropout rate.
            num_nucleotides_original (int): Number of original nucleotide types (e.g., 5 for A,C,G,T,N).
                                           This is used for the DNA prediction head output size.
        """
        super(DNASequenceClassifier, self).__init__()

        self.input_projection = nn.Linear(input_features, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)
        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model

        self.encoder_layers = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])

        # Prediction head for DNA nucleotide reconstruction (e.g., 5 logits for A,C,G,T,N)
        self.dna_prediction_head = nn.Linear(d_model, num_nucleotides_original)
        # Prediction head for RNA expression state prediction (1 logit for binary 0.0 or 1.0)
        self.expression_prediction_head = nn.Linear(d_model, 1)

    def forward(self, src):
        """
        Forward pass for the DNA Sequence Classifier in masked multi-task mode.

        Args:
            src (torch.Tensor): Input sequence tensor.
                                Expected shape: (batch_size, sequence_length, input_features)
        Returns:
            tuple:
                - dna_logits (torch.Tensor): Logits for DNA base prediction (shape: batch_size, seq_length, num_nucleotides_original)
                - expression_logits (torch.Tensor): Logits for expression prediction (shape: batch_size, seq_length, 1)
        """
        src_mask = None # Assuming no explicit padding mask needed if all window_sizes are fixed to SEQ_LENGTH

        src_embedded = self.input_projection(src) * math.sqrt(self.d_model)
        src_embedded = self.dropout(self.positional_encoding(src_embedded))

        enc_output = src_embedded
        for enc_layer in self.encoder_layers:
            # EncoderLayer returns output and attention probabilities, we only need output here
            enc_output, _ = enc_layer(enc_output, src_mask)

        dna_logits = self.dna_prediction_head(enc_output)
        expression_logits = self.expression_prediction_head(enc_output)

        return dna_logits, expression_logits

## 5. Data Loading - Base Dataset (`GenomeExpressionDataset`)

This custom PyTorch `Dataset` handles the initial loading and preprocessing of genomic data. It reads DNA sequences and corresponding RNA expression labels from `.npz` and `.parquet` files. Key features include:
- Loading integer-encoded DNA sequences and float32 expression values.
- Applying reverse complement transformation for sequences on the '-' strand.
- Providing chromosome information, which is crucial for a robust train/validation/test split.

In [10]:
# --- DATASET CLASSES ---

class GenomeExpressionDataset(Dataset):
    """
    Custom Dataset for loading DNA sequence and expression data for genomic regions.
    Loads data from pre-processed .npz and .parquet files.
    Handles reverse complement for '-' strand sequences.
    Returns integer-encoded DNA sequences and float32 expression labels.
    Provides chromosome information for splitting.
    """
    def __init__(self, data_dir):
        self.data_dir = data_dir

        self.data_npz_path = os.path.join(data_dir, 'data.npz')
        self.regions_parquet_path = os.path.join(data_dir, 'regions.parquet')

        try:
            loaded_npz = np.load(self.data_npz_path, allow_pickle=True)
            self.sequence_data = loaded_npz['sequence'] # Stored as integers (0-4)
            self.expression_plus_data = loaded_npz['expressed_plus']
            self.expression_minus_data = loaded_npz['expressed_minus']
            loaded_npz.close()
        except KeyError as e:
            available_keys = list(np.load(self.data_npz_path).keys()) if os.path.exists(self.data_npz_path) else "File not found during key check."
            raise RuntimeError(f"KeyError: Key '{e}' not found in {self.data_npz_path}. "
                               f"Available keys: {available_keys}. "
                               f"Please check your .npz file structure.")
        except Exception as e:
            raise RuntimeError(f"Could not load data from {self.data_npz_path}. Make sure the file exists and is not corrupted: {e}")

        try:
            self.regions_df = pd.read_parquet(self.regions_parquet_path)
            # Ensure 'contig' column exists, as 'chromosome' was not found in previous run
            if 'contig' not in self.regions_df.columns:
                raise ValueError(f"The 'regions.parquet' file must contain a 'contig' column for splitting. 'chromosome' column was not found.")
        except Exception as e:
            raise RuntimeError(f"Could not load regions from {self.regions_parquet_path}. Make sure the file exists and is not corrupted: {e}")

        assert len(self.sequence_data) == len(self.expression_plus_data) == len(self.expression_minus_data)

        # Map for reverse complement: A<->T, C<->G, N<->N (0<->3, 1<->2, 4<->4)
        self.complement_map = np.array([3, 2, 1, 0, 4], dtype=np.uint8)

    def __len__(self):
        return len(self.regions_df)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        region_info = self.regions_df.iloc[idx]

        offset = region_info['offset']
        window_size = region_info['window_size']
        strand = region_info['strand']

        # Get original sequence (integer encoded) and expression labels
        original_nucleotide_segment = self.sequence_data[offset : offset + window_size][::-1].copy()

        if strand == '+':
            original_expression_label_segment = self.expression_plus_data[offset : offset + window_size].copy()
        else: # strand == '-'
            # Apply reverse complement to DNA sequence
            original_nucleotide_segment = self.complement_map[original_nucleotide_segment][::-1].copy()
            # Corresponding expression label segment for the minus strand (already handled by preprocessing)
            original_expression_label_segment = self.expression_minus_data[offset : offset + window_size].copy()

        # Ensure expression_label_segment is a float32 tensor
        original_expression_labels = torch.tensor(original_expression_label_segment, dtype=torch.float32)

        # Return original integer sequence and expression labels
        return torch.tensor(original_nucleotide_segment, dtype=torch.long), original_expression_labels

    def get_all_sample_chromosomes(self):
        """Returns a list of chromosome identifiers for all samples."""
        return self.regions_df['contig'].tolist()

## 6. Data Loading - Masking Dataset (`MultiModalMaskingDataset`)

This is a crucial component that wraps the `GenomeExpressionDataset` to apply the specific masking strategies required for multi-task learning:
- **RNA Masking**: A *contiguous 50%* span of RNA expression values is masked with a special value (`MASK_EXPRESSION_VALUE`).
- **DNA Masking**: *Random 15%* of DNA nucleotides are masked using a BERT-like strategy (80% `[MASK]` token, 10% random nucleotide, 10% original nucleotide).

This dataset prepares the input for the model (`model_input`) and provides the corresponding ground truth labels (`original_nucleotide_labels`, `original_expression_labels`) and target masks (`dna_target_mask`, `rna_target_mask`) indicating which positions need to be predicted.

In [11]:
class MultiModalMaskingDataset(Dataset):
    """
    Wraps GenomeExpressionDataset to apply both contiguous RNA masking (50%)
    and random DNA masking (15%) for a multi-task prediction.
    """
    def __init__(self, base_dataset: GenomeExpressionDataset, seq_length: int,
                 dna_mask_probability=DNA_MASK_PROBABILITY, rna_mask_fraction=RNA_MASK_FRACTION):
        self.base_dataset = base_dataset
        self.seq_length = seq_length # Fixed sequence length from global constant
        self.dna_mask_probability = dna_mask_probability
        self.rna_mask_fraction = rna_mask_fraction

        self.num_nucleotides_original = NUM_NUCLEOTIDES_ORIGINAL
        self.one_hot_nucleotide_dim = ONE_HOT_NUCLEOTIDE_DIM
        self.mask_nucleotide_index = MASK_NUCLEOTIDE_INDEX
        self.mask_expression_value = MASK_EXPRESSION_VALUE

        self.bert_mask_replacement_prob = BERT_MASK_REPLACEMENT_PROB
        self.bert_random_replacement_prob = BERT_RANDOM_REPLACEMENT_PROB
        self.bert_original_replacement_prob = BERT_ORIGINAL_REPLACEMENT_PROB

    def __len__(self):
        return len(self.base_dataset)

    def _one_hot_encode_dna(self, sequence_segment_int, one_hot_dim):
        """
        Helper to one-hot encode nucleotide integer indices for model input.
        Handles the expanded dimension for the MASK token.
        """
        one_hot_tensor = torch.zeros(len(sequence_segment_int), one_hot_dim, dtype=torch.float32)
        # Scatter original nucleotide values (0-4)
        one_hot_tensor.scatter_(1, sequence_segment_int.unsqueeze(1).long(), 1)
        return one_hot_tensor

    def __getitem__(self, idx):
        # Retrieve original integer-encoded DNA sequence and float32 expression labels
        # original_nucleotide_labels: (seq_len,) -> integer indices 0-4
        # original_expression_labels: (seq_len,) -> float32 values 0.0 or 1.0
        original_nucleotide_labels, original_expression_labels = self.base_dataset[idx]

        # Initialize tensors for model input and target masks
        # For model_input, we need to convert original_nucleotide_labels to one-hot
        input_nucleotide_features = self._one_hot_encode_dna(original_nucleotide_labels, self.one_hot_nucleotide_dim) # (seq_len, 6)

        # Input expression state will be (seq_len, 1)
        input_expression_states = original_expression_labels.clone().unsqueeze(-1) # (seq_len, 1)

        # Initialize masks for what we actually need to predict
        dna_target_mask = torch.full((self.seq_length,), False, dtype=torch.bool)
        rna_target_mask = torch.full((self.seq_length,), False, dtype=torch.bool)

        # --- Apply RNA Masking (Contiguous 50%) ---
        rna_mask_span_length = max(1, int(self.seq_length * self.rna_mask_fraction))
        if self.seq_length > rna_mask_span_length:
            rna_mask_start_idx = random.randint(0, self.seq_length - rna_mask_span_length)
        else: # Mask the entire sequence if span length is >= sequence length
            rna_mask_start_idx = 0
            rna_mask_span_length = self.seq_length
        rna_mask_end_idx = rna_mask_start_idx + rna_mask_span_length

        input_expression_states[rna_mask_start_idx : rna_mask_end_idx] = self.mask_expression_value
        rna_target_mask[rna_mask_start_idx : rna_mask_end_idx] = True

        # --- Apply DNA Masking (Random 15% with BERT-like strategy) ---
        for i in range(self.seq_length):
            if random.random() < self.dna_mask_probability:
                dna_target_mask[i] = True # Mark this position for DNA prediction

                rand_val = random.random()
                if rand_val < self.bert_mask_replacement_prob:
                    # 80% of the time: replace with MASK_NUCLEOTIDE_INDEX
                    input_nucleotide_features[i] = torch.zeros(self.one_hot_nucleotide_dim)
                    input_nucleotide_features[i, self.mask_nucleotide_index] = 1.0
                elif rand_val < (self.bert_mask_replacement_prob + self.bert_random_replacement_prob):
                    # 10% of the time: replace with a random original nucleotide (0-4)
                    random_nuc_idx = random.randint(0, self.num_nucleotides_original - 1)
                    input_nucleotide_features[i] = torch.zeros(self.one_hot_nucleotide_dim)
                    input_nucleotide_features[i, random_nuc_idx] = 1.0
                # else (10% of the time): keep original nucleotide (no change to input_nucleotide_features[i] needed)

        # Concatenate nucleotide features with expression states to form model input
        # Resulting shape: (seq_length, ONE_HOT_NUCLEOTIDE_DIM + 1)
        model_input = torch.cat((input_nucleotide_features, input_expression_states), dim=-1)

        # Return:
        # model_input: (seq_len, ONE_HOT_NUCLEOTIDE_DIM + 1) with masked values for model input
        # original_nucleotide_labels: (seq_len,) -> true integer labels (0-4) for DNA prediction target
        # original_expression_labels: (seq_len,) -> true float labels (0.0/1.0) for RNA prediction target
        # dna_target_mask: (seq_len,) -> boolean, True where DNA was masked and needs prediction
        # rna_target_mask: (seq_len,) -> boolean, True where RNA was masked and needs prediction
        return model_input, original_nucleotide_labels, original_expression_labels, dna_target_mask, rna_target_mask

## 7. Data Loading and Chromosome-Based Splitting Logic

This section orchestrates the data loading and prepares the datasets for training, validation, and testing. It dynamically determines the sequence length from the loaded data and implements a robust chromosome-based splitting strategy to ensure that training, validation, and test sets contain data from entirely separate chromosomes. This prevents data leakage and provides a more realistic evaluation of the model's generalization capabilities.

A `DEBUG_DATASET_SIZE` option is included to allow for faster testing with a smaller subset of the data.

In [2]:
# --- DATA LOADING AND SPLITTING LOGIC ---

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Get data directory
data_dir = get_data_dir()

# Initialize base dataset
try:
    base_full_dataset = GenomeExpressionDataset(data_dir)
    print(f"Full base dataset loaded successfully. Total samples: {len(base_full_dataset)}")

    if len(base_full_dataset) > 0:
        # Dynamically determine SEQ_LENGTH (window_size) from the first sample
        sample_dna_int, _ = base_full_dataset[0]
        SEQ_LENGTH = sample_dna_int.shape[0] # Update global variable
        print(f"Detected sequence length (window_size): {SEQ_LENGTH}")
    else:
        print("Warning: Base dataset is empty. Cannot determine SEQ_LENGTH dynamically. Please ensure data is present.")
        exit() # Exit if dataset is empty to prevent errors

except (FileNotFoundError, RuntimeError, ValueError) as e:
    print(f"Error loading base dataset: {e}")
    print("Please ensure your data files ('data.npz', 'regions.parquet') are correctly placed and formatted.")
    exit() # Exit execution if essential files/data are missing or malformed


# --- Determine the working pool of indices based SOLELY on DATA_SUBSET_SIZE ---
initial_pool_indices = list(range(len(base_full_dataset)))
full_dataset_actual_size = len(initial_pool_indices)

if DATA_SUBSET_SIZE is not None:
    if DATA_SUBSET_SIZE < full_dataset_actual_size:
        current_pool_size = max(1, DATA_SUBSET_SIZE) # Ensure at least 1 sample if data exists
        print(f"SUBSET MODE: Creating a random subset of size {current_pool_size} from the full dataset.")
        random.shuffle(initial_pool_indices) # Shuffle the full range of indices
        dataset_to_split_indices = initial_pool_indices[:current_pool_size] # Take the subset
        print(f"Working with a subset of {len(dataset_to_split_indices)} samples for splitting.")
    else: # DATA_SUBSET_SIZE is greater than or equal to the full dataset size
        print("Warning: DATA_SUBSET_SIZE is greater than or equal to the full dataset size. Using the full dataset for splitting (randomly shuffled).")
        random.shuffle(initial_pool_indices) # Shuffle the full dataset
        dataset_to_split_indices = initial_pool_indices # Use all indices, but they are shuffled
        print(f"Working with {len(dataset_to_split_indices)} samples for splitting.")
else: # DATA_SUBSET_SIZE is None
    print("No specific DATA_SUBSET_SIZE set. Using the full dataset for splitting (randomly shuffled).")
    random.shuffle(initial_pool_indices) # Shuffle the full dataset
    dataset_to_split_indices = initial_pool_indices # Use all indices, but they are shuffled
    print(f"Working with {len(dataset_to_split_indices)} samples for splitting.")

# Convert the working pool of indices to a set for efficient lookups
dataset_to_split_indices_set = set(dataset_to_split_indices)


# --- Determine Splitting Method based on USE_CHROMOSOME_SPLIT ---
if USE_CHROMOSOME_SPLIT:
    # Get all chromosome identifiers from the *entire original dataset*
    all_chromosomes_full_dataset = base_full_dataset.get_all_sample_chromosomes()

    # Filter chromosomes to only include those present in the *current pool of samples* (`dataset_to_split_indices`)
    chromosomes_in_current_pool = [all_chromosomes_full_dataset[i] for i in dataset_to_split_indices]
    unique_chromosomes = sorted(list(set(chromosomes_in_current_pool)))
    num_unique_chromosomes = len(unique_chromosomes)

    if num_unique_chromosomes < 3: # Need at least 3 chromosomes for train/val/test split
        print(f"Error: Not enough unique chromosomes in the selected pool ({len(dataset_to_split_indices)} samples) for a chromosome-based split. Found {num_unique_chromosomes}. Falling back to random split.")

        # Fallback to random split on the dataset_to_split_indices
        train_size = int(TRAIN_CHROM_RATIO * len(dataset_to_split_indices))
        val_size = int(VAL_CHROM_RATIO * len(dataset_to_split_indices))
        test_size = len(dataset_to_split_indices) - train_size - val_size

        # Adjust sizes for non-negativity and sum correctness
        if train_size < 0: train_size = 0
        if val_size < 0: val_size = 0
        if test_size < 0: test_size = 0

        total_allocated = train_size + val_size + test_size
        if total_allocated != len(dataset_to_split_indices):
            train_size += (len(dataset_to_split_indices) - total_allocated) # Add/subtract from train to balance
            if train_size < 0: train_size = 0 # Ensure no negative size after adjustment

        train_subset_indices, val_subset_indices, test_subset_indices = random_split(
            dataset_to_split_indices, [train_size, val_size, test_size])
        print(f"Performing random split: Train={len(train_subset_indices)}, Val={len(val_subset_indices)}, Test={len(test_subset_indices)}")
    else: # Enough unique chromosomes, proceed with chromosome-based splitting
        print(f"Found {num_unique_chromosomes} unique chromosomes in the selected pool: {unique_chromosomes}")

        random.shuffle(unique_chromosomes) # Shuffle chromosomes to ensure random assignment

        train_chrom_count = max(1, int(num_unique_chromosomes * TRAIN_CHROM_RATIO))
        val_chrom_count = max(1, int(num_unique_chromosomes * VAL_CHROM_RATIO))

        # Ensure at least one chromosome for test set (and others) if possible with >=3 unique chroms
        if num_unique_chromosomes - train_chrom_count - val_chrom_count < 1 and num_unique_chromosomes >= 3:
            if val_chrom_count > 1: val_chrom_count -= 1
            elif train_chrom_count > 1: train_chrom_count -= 1

        train_chroms = unique_chromosomes[:train_chrom_count]
        val_chroms = unique_chromosomes[train_chrom_count : train_chrom_count + val_chrom_count]
        test_chroms = unique_chromosomes[train_chrom_count + val_chrom_count :]

        # Handle edge cases where a set might be empty due to small number of chromosomes being shuffled
        # This logic tries to ensure each split gets at least one chromosome if possible from remaining
        if not test_chroms and val_chroms: test_chroms.append(val_chroms.pop())
        if not test_chroms and train_chroms: test_chroms.append(train_chroms.pop())
        if not val_chroms and train_chroms and len(train_chroms) > 1 and num_unique_chromosomes >= 2:
             val_chroms.append(train_chroms.pop(0))

        if num_unique_chromosomes >= 3: # Final check to populate empty sets if possible
            if not train_chroms and unique_chromosomes: train_chroms.append(unique_chromosomes.pop(0))
            if not val_chroms and unique_chromosomes: val_chroms.append(unique_chromosomes.pop(0))
            if not test_chroms and unique_chromosomes: test_chroms.append(unique_chromosomes.pop(0))


        # Filter out any potential 'None' values from complex pop() logic in edge cases
        train_chroms = [c for c in train_chroms if c is not None]
        val_chroms = [c for c in val_chroms if c is not None]
        test_chroms = [c for c in test_chroms if c is not None]


        print(f"Chromosome split: Train={train_chroms}, Val={val_chroms}, Test={test_chroms}")

        # Create index subsets based on chromosome assignments
        # 'i in dataset_to_split_indices_set' ensures we only pick from the current working pool of samples
        train_indices = [i for i, chrom in enumerate(all_chromosomes_full_dataset) if chrom in train_chroms and i in dataset_to_split_indices_set]
        val_indices = [i for i, chrom in enumerate(all_chromosomes_full_dataset) if chrom in val_chroms and i in dataset_to_split_indices_set]
        test_indices = [i for i, chrom in enumerate(all_chromosomes_full_dataset) if chrom in test_chroms and i in dataset_to_split_indices_set]

        print(f"Chromosome-based split: Train samples={len(train_indices)}, Val samples={len(val_indices)}, Test samples={len(test_indices)}")

        # Create Subsets (these are now correct indices for the base_full_dataset)
        train_subset_indices = Subset(base_full_dataset, train_indices)
        val_subset_indices = Subset(base_full_dataset, val_indices)
        test_subset_indices = Subset(base_full_dataset, test_indices)

else: # USE_CHROMOSOME_SPLIT is False
    print("Chromosome-based splitting is OFF. Performing random split on the selected samples.")
    train_size = int(TRAIN_CHROM_RATIO * len(dataset_to_split_indices))
    val_size = int(VAL_CHROM_RATIO * len(dataset_to_split_indices))
    test_size = len(dataset_to_split_indices) - train_size - val_size

    # Adjust sizes for non-negativity and sum correctness
    if train_size < 0: train_size = 0
    if val_size < 0: val_size = 0
    if test_size < 0: test_size = 0
    total_allocated = train_size + val_size + test_size
    if total_allocated != len(dataset_to_split_indices):
        train_size += (len(dataset_to_split_indices) - total_allocated)
        if train_size < 0: train_size = 0

    train_subset_indices, val_subset_indices, test_subset_indices = random_split(
        dataset_to_split_indices, [train_size, val_size, test_size])
    print(f"Performing random split: Train={len(train_subset_indices)}, Val={len(val_subset_indices)}, Test={len(test_subset_indices)}")


# Create the masking datasets using the subsets derived from either split method
train_dataset = MultiModalMaskingDataset(train_subset_indices, SEQ_LENGTH,
                                         DNA_MASK_PROBABILITY, RNA_MASK_FRACTION)
val_dataset = MultiModalMaskingDataset(val_subset_indices, SEQ_LENGTH,
                                       DNA_MASK_PROBABILITY, RNA_MASK_FRACTION)
test_dataset = MultiModalMaskingDataset(test_subset_indices, SEQ_LENGTH,
                                        DNA_MASK_PROBABILITY, RNA_MASK_FRACTION)


print(f"Final Dataset sizes: Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}")

# Create DataLoaders
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print("DataLoaders created.")

IndentationError: expected an indented block after 'else' statement on line 52 (2439384809.py, line 55)

## 8. Model Initialization, Loss Functions, and Optimizer

This section initializes the `DNASequenceClassifier` model with the defined hyperparameters. It also sets up the appropriate loss functions for each task and the optimizer for training. Mixed-precision training (`autocast`, `GradScaler`) is configured to improve training speed and memory efficiency on compatible hardware (e.g., NVIDIA GPUs).

In [15]:
# --- MODEL INITIALIZATION, LOSS FUNCTIONS, OPTIMIZER ---

# Initialize model
model = DNASequenceClassifier(
    input_features=NEW_INPUT_FEATURES,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    max_seq_length=SEQ_LENGTH,
    dropout=DROPOUT,
    num_nucleotides_original=NUM_NUCLEOTIDES_ORIGINAL
).to(device)

print(f"Model initialized with {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters.")

# Loss functions
# For DNA (multi-class classification for nucleotides 0-4)
dna_loss_fn = nn.CrossEntropyLoss(reduction='none') # reduction='none' so we can apply mask
# For RNA (binary classification: 0 or 1)
expression_loss_fn = nn.BCEWithLogitsLoss(reduction='none') # reduction='none' for masking

# Optimizer with AdamW for better regularization
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# Learning rate scheduler with warmup
def lr_lambda(current_step: int):
    if current_step < WARMUP_STEPS:
        return float(current_step) / float(max(1, WARMUP_STEPS))
    return 1.0

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# For mixed-precision training
scaler = GradScaler()

print("Model, Loss Functions, Optimizer, and Scheduler configured.")

Model initialized with 399,238 trainable parameters.
Model, Loss Functions, Optimizer, and Scheduler configured.


/var/folders/0b/5d1tzl1x1_x6jr81t655fk0r0000gn/T/ipykernel_27553/203392787.py:35: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/opt/homebrew/anaconda3/envs/ml-base/lib/python3.10/site-packages/torch/amp/grad_scaler.py:136: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


## 9. Training and Validation Functions

This section defines the `train_epoch` and `validate_epoch` functions. These functions encapsulate the core training and validation loop, including:
- Forward pass through the model.
- Calculation of DNA (masked nucleotide) and RNA (masked expression) losses.
- Application of target masks to ensure losses are calculated only for predicted positions.
- Backpropagation and optimization (including mixed-precision scaling).
- Accumulation of gradients over several batches to simulate larger batch sizes (`GRADIENT_ACCUMULATION_STEPS`).
- Calculation of accuracy metrics for both tasks.

In [18]:
# --- TRAINING AND VALIDATION FUNCTIONS ---

def train_epoch(model, dataloader, optimizer, dna_loss_fn, expression_loss_fn, scheduler, scaler, device, accumulation_steps):
    model.train()
    total_dna_loss = 0
    total_rna_loss = 0
    total_combined_loss = 0
    correct_dna_predictions = 0
    total_dna_masked_tokens = 0
    correct_rna_predictions = 0
    total_rna_masked_tokens = 0

    optimizer.zero_grad() # Initialize gradients once per accumulation cycle

    for i, (model_input, original_dna_labels, original_rna_labels, dna_target_mask, rna_target_mask) in enumerate(dataloader):
        model_input = model_input.to(device)
        original_dna_labels = original_dna_labels.to(device) # (batch_size, seq_len)
        original_rna_labels = original_rna_labels.to(device) # (batch_size, seq_len)
        dna_target_mask = dna_target_mask.to(device) # (batch_size, seq_len)
        rna_target_mask = rna_target_mask.to(device) # (batch_size, seq_len)

        with autocast(): # For mixed-precision training
            dna_logits, expression_logits = model(model_input)

            # --- DNA Loss Calculation (Masked Language Modeling) ---
            # Reshape for CrossEntropyLoss: (N, C, ...) where C is num_classes
            # target is (N, ...)
            # dna_logits: (batch_size, seq_len, num_nucleotides_original)
            # original_dna_labels: (batch_size, seq_len)

            # Apply mask to select only relevant positions for loss calculation
            masked_dna_logits = dna_logits[dna_target_mask] # Flattened: (num_masked_dna_tokens, num_nucleotides_original)
            masked_dna_labels = original_dna_labels[dna_target_mask] # Flattened: (num_masked_dna_tokens,)

            if masked_dna_labels.numel() > 0:
                dna_loss = dna_loss_fn(masked_dna_logits, masked_dna_labels)
                dna_loss_scalar = dna_loss.mean() # Get scalar loss for accumulation
            else:
                dna_loss_scalar = torch.tensor(0.0, device=device)
            total_dna_loss += dna_loss_scalar.item()

            # --- RNA Loss Calculation (Binary Classification) ---
            # expression_logits: (batch_size, seq_len, 1)
            # original_rna_labels: (batch_size, seq_len)

            masked_expression_logits = expression_logits[rna_target_mask] # Flattened: (num_masked_rna_tokens, 1)
            masked_rna_labels = original_rna_labels[rna_target_mask] # Flattened: (num_masked_rna_tokens,)

            if masked_rna_labels.numel() > 0:
                # BCEWithLogitsLoss expects (N, *) for input and target
                expression_loss = expression_loss_fn(masked_expression_logits.squeeze(-1), masked_rna_labels)
                expression_loss_scalar = expression_loss.mean() # Get scalar loss for accumulation
            else:
                expression_loss_scalar = torch.tensor(0.0, device=device)
            total_rna_loss += expression_loss_scalar.item()

            combined_loss = dna_loss_scalar + expression_loss_scalar
            total_combined_loss += combined_loss.item()

        # Scale the loss and backpropagate
        scaler.scale(combined_loss / accumulation_steps).backward()

        # Perform optimizer step only after accumulating gradients for 'accumulation_steps' batches
        if (i + 1) % accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad() # Clear gradients for the next accumulation cycle
            scheduler.step() # Update learning rate

        # --- Accuracy Calculation ---
        # DNA Accuracy
        if masked_dna_labels.numel() > 0:
            dna_preds = torch.argmax(masked_dna_logits, dim=-1)
            correct_dna_predictions += (dna_preds == masked_dna_labels).sum().item()
            total_dna_masked_tokens += masked_dna_labels.numel()

        # RNA Accuracy
        if masked_rna_labels.numel() > 0:
            rna_preds = (torch.sigmoid(masked_expression_logits) > 0.5).float().squeeze(-1)
            correct_rna_predictions += (rna_preds == masked_rna_labels).sum().item()
            total_rna_masked_tokens += masked_rna_labels.numel()

    # Handle remaining gradients if the last batch didn't complete an accumulation cycle
    if (i + 1) % accumulation_steps != 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        scheduler.step() # Make sure scheduler is stepped even if not a full accumulation cycle

    avg_combined_loss = total_combined_loss / len(dataloader)
    avg_dna_loss = total_dna_loss / len(dataloader)
    avg_rna_loss = total_rna_loss / len(dataloader)
    dna_accuracy = correct_dna_predictions / total_dna_masked_tokens if total_dna_masked_tokens > 0 else 0
    rna_accuracy = correct_rna_predictions / total_rna_masked_tokens if total_rna_masked_tokens > 0 else 0

    return avg_combined_loss, avg_dna_loss, avg_rna_loss, dna_accuracy, rna_accuracy

def validate_epoch(model, dataloader, dna_loss_fn, expression_loss_fn, device):
    model.eval()
    total_dna_loss = 0
    total_rna_loss = 0
    total_combined_loss = 0
    correct_dna_predictions = 0
    total_dna_masked_tokens = 0
    correct_rna_predictions = 0
    total_rna_masked_tokens = 0

    with torch.no_grad():
        for model_input, original_dna_labels, original_rna_labels, dna_target_mask, rna_target_mask in dataloader:
            model_input = model_input.to(device)
            original_dna_labels = original_dna_labels.to(device)
            original_rna_labels = original_rna_labels.to(device)
            dna_target_mask = dna_target_mask.to(device)
            rna_target_mask = rna_target_mask.to(device)

            dna_logits, expression_logits = model(model_input)

            # --- DNA Loss Calculation ---
            masked_dna_logits = dna_logits[dna_target_mask]
            masked_dna_labels = original_dna_labels[dna_target_mask]

            if masked_dna_labels.numel() > 0:
                dna_loss = dna_loss_fn(masked_dna_logits, masked_dna_labels)
                total_dna_loss += dna_loss.mean().item()

                dna_preds = torch.argmax(masked_dna_logits, dim=-1)
                correct_dna_predictions += (dna_preds == masked_dna_labels).sum().item()
                total_dna_masked_tokens += masked_dna_labels.numel()

            # --- RNA Loss Calculation ---
            masked_expression_logits = expression_logits[rna_target_mask]
            masked_rna_labels = original_rna_labels[rna_target_mask]

            if masked_rna_labels.numel() > 0:
                expression_loss = expression_loss_fn(masked_expression_logits.squeeze(-1), masked_rna_labels)
                total_rna_loss += expression_loss.mean().item()

                rna_preds = (torch.sigmoid(masked_expression_logits) > 0.5).float().squeeze(-1)
                correct_rna_predictions += (rna_preds == masked_rna_labels).sum().item()
                total_rna_masked_tokens += masked_rna_labels.numel()

            total_combined_loss += (dna_loss.mean().item() if masked_dna_labels.numel() > 0 else 0) + \
                                   (expression_loss.mean().item() if masked_rna_labels.numel() > 0 else 0)

    avg_combined_loss = total_combined_loss / len(dataloader)
    avg_dna_loss = total_dna_loss / len(dataloader)
    avg_rna_loss = total_rna_loss / len(dataloader)
    dna_accuracy = correct_dna_predictions / total_dna_masked_tokens if total_dna_masked_tokens > 0 else 0
    rna_accuracy = correct_rna_predictions / total_rna_masked_tokens if total_rna_masked_tokens > 0 else 0

    return avg_combined_loss, avg_dna_loss, avg_rna_loss, dna_accuracy, rna_accuracy

## 10. Main Training Loop and Visualization

This section contains the main training loop, which iterates over the defined number of epochs. For each epoch, it calls the `train_epoch` and `validate_epoch` functions, records the losses and accuracies, and then visualizes the training progress over time.

In [20]:
# --- MAIN TRAINING LOOP ---

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

train_dna_losses = []
val_dna_losses = []
train_rna_losses = []
val_rna_losses = []

train_dna_accuracies = []
val_dna_accuracies = []
train_rna_accuracies = []
val_rna_accuracies = []

train_total_losses = []
val_total_losses = []

best_val_loss = float('inf')

print("Starting training...")
for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")

    # Training step
    (combined_train_loss, dna_train_loss, rna_train_loss, 
     dna_train_accuracy, rna_train_accuracy) = train_epoch(
        model, train_dataloader, optimizer, dna_loss_fn, expression_loss_fn,
        scheduler, scaler, device, GRADIENT_ACCUMULATION_STEPS
    )

    # Validation step
    (combined_val_loss, dna_val_loss, rna_val_loss, 
     dna_val_accuracy, rna_val_accuracy) = validate_epoch(
        model, val_dataloader, dna_loss_fn, expression_loss_fn, device
    )

    print(f"Train Combined Loss: {combined_train_loss:.4f}, DNA Loss: {dna_train_loss:.4f}, RNA Loss: {rna_train_loss:.4f}")
    print(f"Train DNA Accuracy: {dna_train_accuracy:.4f}, RNA Accuracy: {rna_train_accuracy:.4f}")
    print(f"Val Combined Loss: {combined_val_loss:.4f}, DNA Loss: {dna_val_loss:.4f}, RNA Loss: {rna_val_loss:.4f}")
    print(f"Val DNA Accuracy: {dna_val_accuracy:.4f}, RNA Accuracy: {rna_val_accuracy:.4f}")

    # Store metrics
    train_total_losses.append(combined_train_loss)
    val_total_losses.append(combined_val_loss)
    train_dna_losses.append(dna_train_loss)
    val_dna_losses.append(dna_val_loss)
    train_rna_losses.append(rna_train_loss)
    val_rna_losses.append(rna_val_loss)
    train_dna_accuracies.append(dna_train_accuracy)
    val_dna_accuracies.append(dna_val_accuracy)
    train_rna_accuracies.append(rna_train_accuracy)
    val_rna_accuracies.append(rna_val_accuracy)

    # Save the best model based on validation loss
    if combined_val_loss < best_val_loss:
        best_val_loss = combined_val_loss
        torch.save(model.state_dict(), 'best_model.pth')
        print("Saved best model to best_model.pth")

print("\nTraining complete.")

# Plotting Training and Validation Metrics ---

epochs_range = range(1, NUM_EPOCHS + 1)

plt.figure(figsize=(18, 5)) # Wider figure for 3 plots

# Plot Total Loss
plt.subplot(1, 3, 1)
plt.plot(epochs_range, train_total_losses, label='Training Total Loss')
plt.plot(epochs_range, val_total_losses, label='Validation Total Loss')
plt.title('Training and Validation Total Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot DNA Accuracy
plt.subplot(1, 3, 2)
plt.plot(epochs_range, train_dna_accuracies, label='Training DNA Accuracy')
plt.plot(epochs_range, val_dna_accuracies, label='Validation DNA Accuracy')
plt.title('Training and Validation DNA Accuracy (Masked)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plot RNA Accuracy
plt.subplot(1, 3, 3)
plt.plot(epochs_range, train_rna_accuracies, label='Training RNA Accuracy')
plt.plot(epochs_range, val_rna_accuracies, label='Validation RNA Accuracy')
plt.title('Training and Validation RNA Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

Starting training...

Epoch 1/50


Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/opt/homebrew/anaconda3/envs/ml-base/lib/python3.10/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/opt/homebrew/anaconda3/envs/ml-base/lib/python3.10/multiprocessing/spawn.py", line 126, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: Can't get attribute 'MultiModalMaskingDataset' on <module '__main__' (built-in)>


KeyboardInterrupt: 